In [2]:
import os
import cv2
import albumentations as A

In [11]:
# ==========================================
# 1. CONFIGURATION & DIRECTORIES
# ==========================================
DATASETS_ROOT = "New-Datasets" # Where your Folder_A, Folder_B, etc., live
FINAL_EXPORT_DIR = "Pre-Processed-Dataset" # The unified output folder

# YOLOv5/v8 Output Structure
FINAL_IMAGES_DIR = os.path.join(FINAL_EXPORT_DIR, "images/train")
FINAL_LABELS_DIR = os.path.join(FINAL_EXPORT_DIR, "labels/train")
os.makedirs(FINAL_IMAGES_DIR, exist_ok=True)
os.makedirs(FINAL_LABELS_DIR, exist_ok=True)

# Cleansing Parameters
VALID_CLASSES = {0, 1, 2}
MIN_PIXELS = 10
DECIMAL_PLACES = 6

In [12]:
# ==========================================
# 2. ALBUMENTATIONS STRATEGY
# ==========================================
# These filters apply to EVERY image to bridge the sensor/lighting gap
global_augs = [
    A.MotionBlur(blur_limit=(3, 5), p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3)
]

# These are the surgical filters for specific folder biases
surgical_augs = {
    "studio_bias": [
        # Uses the new range tuples for holes and dimensions
        A.CoarseDropout(num_holes_range=(1, 6), hole_height_range=(10, 48), hole_width_range=(10, 48), p=0.5), 
        A.HueSaturationValue(hue_shift_limit=40, sat_shift_limit=40, p=0.6), 
        A.ToGray(p=0.2)
    ],
    "low_res_crowd": [
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.6), 
        A.RandomBrightnessContrast(contrast_limit=0.4, p=0.5)  
    ],
    "dashcam_cctv": [
        # Uses std_range (as a fraction) instead of var_limit
        A.GaussNoise(std_range=(0.2, 0.4), p=0.6), 
        # Uses quality_range instead of lower/upper
        A.ImageCompression(quality_range=(50, 75), p=0.7) 
    ],
    "default": [] 
}

# Map your specific folders to their surgical strategy
folder_mapping = {
    "Youtube-scraped-v3": "low_res_crowd",
    "gender.multiclass-turban-120": "studio_bias",
    "gender.multiclass-hijab-200": "studio_bias",
    "Random-images-no-demographic-800": "low_res_crowd",
    "IRD-250-v2": "dashcam_cctv",
    "Github-scraped-useful-600": "low_res_crowd",
    "Front-View-Turban": "studio_bias"
}

In [15]:
# ==========================================
# 3. MASTER EXECUTION LOOP
# ==========================================
print("🚀 Starting Master Data Pipeline...")
stats = {"original_saved": 0, "augmented_saved": 0, "dropped_corrupt": 0, "dropped_tiny": 0}

for dataset_name in sorted(os.listdir(DATASETS_ROOT)):
    dataset_path = os.path.join(DATASETS_ROOT, dataset_name)
    images_dir = os.path.join(dataset_path, "images/train")
    labels_dir = os.path.join(dataset_path, "labels/train")
    
    if not os.path.isdir(dataset_path) or not os.path.exists(images_dir):
        continue
        
    print(f"\nProcessing: {dataset_name}...")
    
    # Dynamically build this folder's specific pipeline (Global + Surgical)
    strategy_key = folder_mapping.get(dataset_name, "default")
    combined_filters = global_augs + surgical_augs[strategy_key]
    
    # min_visibility=0.2 ensures if CoarseDropout covers 80% of a box, it drops the label safely
    transform = A.Compose(combined_filters, bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.2))

    for img_file in os.listdir(images_dir):
        if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
            
        base_name = os.path.splitext(img_file)[0]
        img_path = os.path.join(images_dir, img_file)
        txt_path = os.path.join(labels_dir, base_name + ".txt")
        
        # 1. Orphan Check
        if not os.path.exists(txt_path):
            continue
            
        # 2. Corrupt Image Check
        image = cv2.imread(img_path)
        if image is None:
            stats["dropped_corrupt"] += 1
            continue
            
        img_height, img_width = image.shape[:2]
        
        # 3. Label Cleansing & Boundary Capping
        clean_bboxes = []
        clean_classes = []
        
        with open(txt_path, "r") as f:
            for line in f.readlines():
                try:
                    parts = line.split()
                    if len(parts) < 5: continue
                        
                    class_id = int(parts[0])
                    if class_id not in VALID_CLASSES: continue
                        
                    cx, cy, w, h = map(float, parts[1:5])
                    
                    # Micro-box prune
                    if (w * img_width) < MIN_PIXELS or (h * img_height) < MIN_PIXELS:
                        stats["dropped_tiny"] += 1
                        continue
                        
                    # Boundary cap
                    x_min = max(0.0, cx - (w / 2.0))
                    y_min = max(0.0, cy - (h / 2.0))
                    x_max = min(1.0, cx + (w / 2.0))
                    y_max = min(1.0, cy + (h / 2.0))
                    
                    new_w = x_max - x_min
                    new_h = y_max - y_min
                    new_cx = x_min + (new_w / 2.0)
                    new_cy = y_min + (new_h / 2.0)
                    
                    clean_bboxes.append([new_cx, new_cy, new_w, new_h])
                    clean_classes.append(class_id)
                except ValueError:
                    continue
        
        # If all boxes were dropped (tiny/invalid), skip the image
        if not clean_bboxes:
            continue
            
        # 4. EXPORT ORIGINAL (Cleaned & Renamed to prevent collision)
        # E.g., Folder_A_image1.jpg
        export_base_name = f"{dataset_name}_{base_name}"
        
        orig_img_export = os.path.join(FINAL_IMAGES_DIR, f"{export_base_name}.jpg")
        orig_txt_export = os.path.join(FINAL_LABELS_DIR, f"{export_base_name}.txt")
        
        cv2.imwrite(orig_img_export, image)
        with open(orig_txt_export, "w") as f:
            for cls, bbox in zip(clean_classes, clean_bboxes):
                f.write(f"{cls} {bbox[0]:.{DECIMAL_PLACES}f} {bbox[1]:.{DECIMAL_PLACES}f} {bbox[2]:.{DECIMAL_PLACES}f} {bbox[3]:.{DECIMAL_PLACES}f}\n")
        stats["original_saved"] += 1
        
        # 5. GENERATE & EXPORT AUGMENTED VARIANT
        try:
            augmented = transform(image=image, bboxes=clean_bboxes, class_labels=clean_classes)
            
            aug_img_export = os.path.join(FINAL_IMAGES_DIR, f"{export_base_name}_aug.jpg")
            aug_txt_export = os.path.join(FINAL_LABELS_DIR, f"{export_base_name}_aug.txt")
            
            # Only save if bounding boxes survived the transformation
            if len(augmented['bboxes']) > 0:
                cv2.imwrite(aug_img_export, augmented['image'])
                with open(aug_txt_export, "w") as f:
                    for cls, bbox in zip(augmented['class_labels'], augmented['bboxes']):
                        f.write(f"{cls} {bbox[0]:.{DECIMAL_PLACES}f} {bbox[1]:.{DECIMAL_PLACES}f} {bbox[2]:.{DECIMAL_PLACES}f} {bbox[3]:.{DECIMAL_PLACES}f}\n")
                stats["augmented_saved"] += 1
        except Exception as e:
            # Catch Albumentations math errors if a box gets squeezed to 0 width
            pass

print("\n" + "="*45)
print(" 🏁 PIPELINE COMPLETE")
print("="*45)
print(f"Clean Originals Exported: {stats['original_saved']}")
print(f"Augmented Variants Exported: {stats['augmented_saved']}")
print(f"Total Final Dataset Size: {stats['original_saved'] + stats['augmented_saved']} images")
print("-" * 45)
print(f"Corrupt Files Dropped: {stats['dropped_corrupt']}")
print(f"Micro-Boxes (<{MIN_PIXELS}px) Pruned: {stats['dropped_tiny']}")
print(f"\nYour YOLO-ready dataset is waiting in: ./{FINAL_EXPORT_DIR}/")

🚀 Starting Master Data Pipeline...

Processing: Front-View-Turban...

Processing: Github-scraped-useful-600...

Processing: IRD-250-v2...

Processing: Random-images-no-demographic-800...

Processing: Youtube-scraped-v3...

Processing: gender.multiclass-hijab-200...

Processing: gender.multiclass-turban-120...

 🏁 PIPELINE COMPLETE
Clean Originals Exported: 2591
Augmented Variants Exported: 2591
Total Final Dataset Size: 5182 images
---------------------------------------------
Corrupt Files Dropped: 0
Micro-Boxes (<10px) Pruned: 16

Your YOLO-ready dataset is waiting in: ./Pre-Processed-Dataset/
